# 03 Models

Runs from the README, each in its own section:

| Run | Method | Status |
|---|---|---|
| Baseline | Most popular beers in train | done |
| B | BPR | to do |
| C | B + home-country bonus | to do |

Every run is scored with the same `evaluate()` on validation and test users.

In [ ]:
import os
from pathlib import Path

# Run from the project folder so `src` can be imported
if Path.cwd().name == "notebooks":
    os.chdir("..")

## Setup (shared by all runs)

In [ ]:
import pandas as pd

from src.metrics import evaluate
from src.models import popular_beers, recommend_popular
from src.split import load_split

train, history, hidden = load_split()

# Coverage is measured against every beer the models can recommend
n_beers = train["beer_id"].nunique()
GROUPS = ["validation", "test"]
results = []

In [ ]:
def score_run(run_name, recommendations):
    """Evaluate a run on validation and test users and add it to the results."""
    for group in GROUPS:
        group_hidden = hidden[hidden["group"] == group]
        scores = evaluate(recommendations, group_hidden, n_beers)
        results.append({"run": run_name, "group": group, **scores})
    return pd.DataFrame(results).query("run == @run_name")

## Baseline: most popular beers

Rank beers by number of ratings from users outside Norway. Every user gets the same list, minus beers they already rated.

In [ ]:
ranking = popular_beers(train)
baseline_recommendations = recommend_popular(ranking, history)
score_run("baseline", baseline_recommendations)

The 10 most popular beers:

In [ ]:
from src.data import load_beers

beer_info = load_beers().set_index("beer_id")[["beer_name", "style", "brewery_name"]]
beer_info.loc[ranking[:10]]

## Save results

In [ ]:
pd.DataFrame(results).to_csv("results/model_results.csv", index=False)
pd.DataFrame(results)